In [1]:
import kagglehub
import numpy as np
import pandas as pd
import os
import time
import multiprocessing as mp
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score,recall_score,roc_auc_score
from itertools import product


In [43]:
# Descargar el conjunto de datos completo
# Esto devolverá la ruta del directorio local donde se ha descargado el conjunto de datos.
dataset_root_path = kagglehub.dataset_download(
    "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
)


Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.


#Reducción de datos
Vamos a reducir el dataset original de a 5k imagenes

In [3]:
import pandas as pd

train_csv = os.path.join(dataset_root_path, "Train.csv")

df = pd.read_csv(train_csv)

print(df.head())
print(df.shape)

   Width  Height  Roi.X1  Roi.Y1  Roi.X2  Roi.Y2  ClassId  \
0     27      26       5       5      22      20       20   
1     28      27       5       6      23      22       20   
2     29      26       6       5      24      21       20   
3     28      27       5       6      23      22       20   
4     28      26       5       5      23      21       20   

                             Path  
0  Train/20/00020_00000_00000.png  
1  Train/20/00020_00000_00001.png  
2  Train/20/00020_00000_00002.png  
3  Train/20/00020_00000_00003.png  
4  Train/20/00020_00000_00004.png  
(39209, 8)


In [25]:
df_muestra = df.sample(n=30000, random_state=42)

print(df_muestra.shape)

(30000, 8)


#Preprocesamiento

In [26]:
from PIL import Image

X = []
y = []

#iteramos sobre el df para cargar, redimensionar y coleccionar las imagenes y etiquetas
for index, row in df_muestra.iterrows():
    # construimos la ruta completa hacia las imagenes
    image_path = os.path.join(dataset_root_path, row['Path'])

    try:
        #cargar la imagen
        img = Image.open(image_path)
        img = img.resize((96, 96)) #primero fue de 32px-> 64px, ->96px
        #Convertimos la imagen a un arreglo para integrarla a X
        X.append(np.array(img))
        #Integramos ClassId a y
        y.append(row['ClassId'])
    except Exception as e:
        print(f"Error procesando {image_path}: {e}")

# Convertir las listas a arreglos
X = np.array(X)
y = np.array(y)

print(f"Dimensiones de imagenes procesadas (X): {X.shape}")
print(f"Dimensiones de etiquetas (y): {y.shape}")

Dimensiones de imagenes procesadas (X): (30000, 96, 96, 3)
Dimensiones de etiquetas (y): (30000,)


In [27]:
import cv2
#aplicaremos filtro gaussiano a cada imagen en X. Inicialmente se aplicó (5,5) pero no dio buenos resultados, reducimos a (3,3)
#El valor 0 indica que la desviación estándar en las direcciones X e Y se calcula a partir del tamaño del kernel
X_smoothed = np.array([cv2.GaussianBlur(img, (3, 3), 0) for img in X])

print(f"dimensiones de imagenes suavizadas(X_smoothed): {X_smoothed.shape}")

dimensiones de imagenes suavizadas(X_smoothed): (30000, 96, 96, 3)


### SIFT

Es más comun realizar sift sobre grises perp considerando que el color es importante para este conjunto ya que colores como amarillo, rojos y blancos son de importancia visual va a dejar esta configuracion. Inicialmente sí se hizo sobre grises pero no dio buenos resultados


In [28]:
import cv2

#inicializar sift
sift = cv2.SIFT_create()

all_descriptors = []

for i, img_gray in enumerate(X_smoothed):
    # Detectar keypoints y descriptores
    keypoints, descriptors = sift.detectAndCompute(img_gray, None)

    if descriptors is not None:
        all_descriptors.append(descriptors)

# Concatenar todos los descriptores en un único array NumPy
# Este array se utilizará para entrenar el vocabulario BOVW

if all_descriptors:
    all_descriptors_np = np.vstack(all_descriptors)
    print(f"Total SIFT descriptores extraídos: {all_descriptors_np.shape}")
else:
    all_descriptors_np = np.array([])
    print("No se extrayeron descriptores SIFT.")

Total SIFT descriptores extraídos: (943903, 128)


### Creación de vocabulario de bolsa de palabras visuales (BoVW)
Ahora, crearemos un vocabulario visual agrupando los descriptores SIFT mediante K-Means

In [33]:
from sklearn.cluster import MiniBatchKMeans

#definimos el numero de palabras visuales (clusters)
k = 2000

if all_descriptors_np.shape[0] > 0:
   # Usamos MiniBatchKMeans para mayor eficiencia con grandes conjuntos de datos
   # Es una alternativa más rápida a KMeans
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, n_init='auto', verbose=False)
    kmeans.fit(all_descriptors_np)
    visual_vocabulary = kmeans.cluster_centers_
    print(f"Dimensiones del vocabulario visual: {visual_vocabulary.shape}")
else:
    print("No se pudo crear vocabulario,sin descriptores.")
    visual_vocabulary = None

Dimensiones del vocabulario visual: (2000, 128)


### Generación de vectores de características BoVW
Tras crear el vocabulario, representaremos cada imagen como un histograma de estas palabras visuales. Este histograma cuenta cuántas veces aparece cada palabra visual en una imagen.

In [34]:
if visual_vocabulary is not None:
    # Inicializar BOWImgDescriptorExtractor
    # Utiliza el detector SIFT y el vocabulario KMeans para generar histogramas
    bow_extractor = cv2.BOWImgDescriptorExtractor(sift, cv2.BFMatcher(cv2.NORM_L2))
    bow_extractor.setVocabulary(visual_vocabulary)

    # Generar vectores de características BoVW para cada imagen
    bovw_features = []
    for img_gray in X_smoothed:
        keypoints = sift.detect(img_gray, None)
        # Calcula el histograma BoVW para la imagen
        if keypoints:
            features = bow_extractor.compute(img_gray, keypoints)
            if features is not None:
                bovw_features.append(features.flatten())
            else:
                # Si el cálculo devuelve None, agregue un vector cero
                bovw_features.append(np.zeros(k))
        else:
            #Si no se detectan puntos clave, agregue un vector cero.
            bovw_features.append(np.zeros(k))

    bovw_features_np = np.array(bovw_features)
    print(f"Características BoVW generadas con dimension: {bovw_features_np.shape}")
else:
    bovw_features_np = None
    print("No se pueden generar las características de BoVW, falta el vocabulario.")

Características BoVW generadas con dimension: (30000, 2000)


### Análisis de Componentes Principales (PCA)
Finalmente, aplicaremos PCA para reducir la dimensionalidad de los vectores de características BoVW, lo que puede ayudar a mejorar el rendimiento del modelo y reducir el tiempo de cálculo, especialmente si el espacio de características es muy grande

In [35]:
from sklearn.decomposition import PCA

if bovw_features_np is not None and bovw_features_np.shape[0] > 0:
  # Inicializar el PCA, conservando el 95% de la varianza
  # También se puede especificar un número fijo de componentes, por ejemplo, n_components=50
    pca = PCA(n_components=500, random_state=42)

    #Adaptar PCA a las características de BoVW y transformarlas
    bovw_features_pca = pca.fit_transform(bovw_features_np)

    print(f"características BoVW después de PCA: {bovw_features_pca.shape}")
    print(f"número de componentes seleccionados por PCA: {pca.n_components_}")
else:
    bovw_features_pca = None
    print("No se pudo aplicar PCA. Sin características BoVW .")

características BoVW después de PCA: (30000, 500)
número de componentes seleccionados por PCA: 500


# RF

In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(bovw_features_pca, y, test_size=0.2, random_state=42)

# Paralelo Random Forest
def rb_train(data):
    X_sub, y_sub = data
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
    model.fit(X_sub, y_sub)
    return model

tiempos_rf = []
accura_rf = []

# Número de procesos
NUM_PROCESOS_MAX = 7

for i in range(2, NUM_PROCESOS_MAX + 1):
    # dividir dataset para el número actual de procesadores 'i'
    indices = np.array_split(np.arange(len(X_train)), i)
    data_splits = [(X_train[idx], y_train[idx]) for idx in indices]

    start_par_rf = time.time()

    with mp.Pool(processes=i) as pool:
        rf_models = pool.map(rb_train, data_splits)

    t_par_rf = time.time() - start_par_rf

    # usar el primer modelo para evaluar
    pred_par_rf = rf_models[0].predict(X_test)
    acc_par_rf = accuracy_score(y_test, pred_par_rf)

    print(f"Procesadores: {i}")
    print(f"Tiempo paralelo RF: {t_par_rf:.2f} segundos")
    print(f"Accuracy RF (usando un modelo): {acc_par_rf:.4f}")
    print("\n")

    tiempos_rf.append(t_par_rf)
    accura_rf.append(acc_par_rf)

print("Tiempos de ejecución paralela (RF):")
print(tiempos_rf)
print("Accuracys paralelas (RF):")
print(accura_rf)


Procesadores: 2
Tiempo paralelo RF: 31.08 segundos
Accuracy RF (usando un modelo): 0.6790


Procesadores: 3
Tiempo paralelo RF: 19.76 segundos
Accuracy RF (usando un modelo): 0.6657


Procesadores: 4
Tiempo paralelo RF: 14.39 segundos
Accuracy RF (usando un modelo): 0.6420


Procesadores: 5
Tiempo paralelo RF: 11.29 segundos
Accuracy RF (usando un modelo): 0.6355


Procesadores: 6
Tiempo paralelo RF: 9.32 segundos
Accuracy RF (usando un modelo): 0.6230


Procesadores: 7
Tiempo paralelo RF: 8.05 segundos
Accuracy RF (usando un modelo): 0.6125


Tiempos de ejecución paralela (RF):
[31.08391046524048, 19.75647473335266, 14.390368223190308, 11.294682741165161, 9.324719429016113, 8.045414686203003]
Accuracys paralelas (RF):
[0.679, 0.6656666666666666, 0.642, 0.6355, 0.623, 0.6125]


In [42]:
# Calcular tiempo de entrenamiento secuencial para Random Forest
start_seq_rf = time.time()
sequential_rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
sequential_rf_model.fit(X_train, y_train)
t_seq = time.time() - start_seq_rf

print(f"Tiempo secuencial de entrenamiento RF: {t_seq:.2f} segundos\n")

# Número de procesos
NUM_PROCESOS_MAX = 7

for i in range(len(tiempos_rf)):
    num_procs = i + 2 # Procesos de 2 a NUM_PROCESOS_MAX
    t_par_i = tiempos_rf[i]

    speed_up = t_seq / t_par_i
    efficiency = speed_up / num_procs

    print(f"Para {num_procs} procesos:")
    print(f"  Tiempo paralelo: {t_par_i:.2f} segundos")
    print(f"  Speed Up: {speed_up:.2f}x")
    print(f"  Eficiencia: {efficiency:.2f}\n")


Tiempo secuencial de entrenamiento RF: 65.08 segundos

Para 2 procesos:
  Tiempo paralelo: 31.08 segundos
  Speed Up: 2.09x
  Eficiencia: 1.05

Para 3 procesos:
  Tiempo paralelo: 19.76 segundos
  Speed Up: 3.29x
  Eficiencia: 1.10

Para 4 procesos:
  Tiempo paralelo: 14.39 segundos
  Speed Up: 4.52x
  Eficiencia: 1.13

Para 5 procesos:
  Tiempo paralelo: 11.29 segundos
  Speed Up: 5.76x
  Eficiencia: 1.15

Para 6 procesos:
  Tiempo paralelo: 9.32 segundos
  Speed Up: 6.98x
  Eficiencia: 1.16

Para 7 procesos:
  Tiempo paralelo: 8.05 segundos
  Speed Up: 8.09x
  Eficiencia: 1.16

